# Prediction de la RUL (UC5) - Jalon 1 : MLP (flotte A)

Flux du notebook :

1. Analyse exploratoire (EDA) et choix des entrees
2. Preparation des donnees (plafonnement de la RUL, cible, normalisation par regime et globale)
3. Metriques, journal d'experiences et fabriques de modeles
4. Modele de base (9 entrees) : entrainement et evaluation
5. Modele a 3 entrees vs modele de base -> **3 entrees retenus**
6. Normalisation globale vs regime (3 entrees) -> **regime retenu**
7. Comparaison des fonctions de perte (3 entrees, regime) : MSE vs asymetrique


In [ ]:
# --- Bibliotheques et reproductibilite ---
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

print("tensorflow :", tf.__version__, "| keras :", keras.__version__, "| numpy :", np.__version__)


In [ ]:
# --- Constantes et dossier des donnees (flotte A, type A) ---
DATA = Path("../data_UC5/uc5_flotteA") if Path.cwd().name == "notebooks" else Path("data_UC5/uc5_flotteA")

RUL_MAX = 125        # la RUL est plafonnee a 125 cycles
SEUIL_ALERTE = 30    # une pompe est signalee si la RUL predite < 30
SEUIL_FA = 60        # fausse alerte si la RUL reelle > 60
DECOUPES = [5, 15, 30, 45, 60, 90, 120]   # points de coupe pour anticipation / fausses alertes

CAPTEURS = [f"capteur_{i}" for i in range(8)]
FEATURES = CAPTEURS + ["charge"]                    # modele de base : 9 entrees
FEATURES_3 = ["capteur_4", "capteur_5", "charge"]   # modele retenu : 3 entrees

print("dossier data :", DATA.resolve())
print("entrees de base :", FEATURES)
print("entrees reduites :", FEATURES_3)


## 1. Analyse exploratoire (EDA)

In [ ]:
# --- Chargement des tableaux .npy de la flotte A ---
def charger(split):
    """Charge un decoupage (train/val/test) dans un DataFrame (un cycle par ligne)."""
    capteurs = np.load(DATA / f"capteurs_{split}.npy")          # (N, 8)
    charge = np.round(np.load(DATA / f"charge_{split}.npy").astype(float), 1)
    cycle = np.load(DATA / f"cycle_{split}.npy")                # (N,)
    pompe = np.load(DATA / f"pompe_{split}.npy")                # (N,)
    d = pd.DataFrame({"pompe": pompe, "cycle": cycle, "charge": charge})
    d = pd.concat([d, pd.DataFrame(capteurs, columns=CAPTEURS)], axis=1)
    try:
        d["rul"] = np.load(DATA / f"rul_{split}.npy")           # absent du jeu de test
    except FileNotFoundError:
        pass
    return d.sort_values(["pompe", "cycle"]).reset_index(drop=True)

TA = {s: charger(s) for s in ["train", "val", "test"]}
print({s: TA[s].shape for s in TA})


In [ ]:
# --- Les 5 premieres lignes du train (un cycle par ligne) ---
TA["train"].head()


In [ ]:
# --- Repartition des pompes et des cycles ---
resume = pd.DataFrame({
    "pompes": {s: TA[s]["pompe"].nunique() for s in TA},
    "cycles": {s: len(TA[s]) for s in TA},
    "cycles_max_par_pompe": {s: int(TA[s].groupby("pompe")["cycle"].max().max()) for s in TA},
})
print("decoupage impose par pompe (aucun recouvrement) :",
      set(TA["train"]["pompe"]) & set(TA["val"]["pompe"]))
resume


In [ ]:
# --- Repartition des regimes de charge (train) ---
TA["train"]["charge"].value_counts()


In [ ]:
# --- Statistiques descriptives des capteurs (train) ---
TA["train"][CAPTEURS].describe().T


In [ ]:
# --- Distribution des capteurs ---
TA["train"][CAPTEURS].hist(figsize=(14, 7), bins=40)
plt.suptitle("Distribution des capteurs (train, flotte A)")
plt.tight_layout()
plt.show()


In [ ]:
# --- Correlation capteur / RUL : sert a selectionner les capteurs utiles ---
corr = TA["train"][CAPTEURS + ["rul"]].corr()["rul"].drop("rul").sort_values()

plt.figure(figsize=(8, 4))
sns.barplot(x=corr.values, y=corr.index, hue=corr.index, palette="vlag", legend=False)
for j, v in enumerate(corr.values):
    plt.text(v + (0.01 if v >= 0 else -0.01), j, f"{v:.2f}",
             va="center", ha="left" if v >= 0 else "right", fontsize=8)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation capteur / RUL (train, flotte A)")
plt.xlabel("correlation de Pearson")
plt.tight_layout()
plt.show()


Les capteurs **capteur_4** et **capteur_5** presentent la meilleure correlation (en valeur absolue) avec la RUL, et le regime de charge **charge** module fortement les mesures : ces 3 variables forment le jeu reduit `FEATURES_3`.

In [ ]:
# --- Effet du regime de charge sur les capteurs ---
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, c in zip(axes, CAPTEURS[:4]):
    sns.boxplot(data=TA["train"], x="charge", y=c, ax=ax)
plt.suptitle("Effet du regime de charge sur les 4 premiers capteurs")
plt.tight_layout()
plt.show()


## 2. Preparation des donnees

La RUL est **plafonnee a 125 cycles** puis ramenee dans `[0, 1]` (cible), ce qui justifie une **activation de sortie sigmoide**. Les capteurs sont normalises de deux facons pour l'ablation : **par regime de charge** (statistiques des cycles sains) et **globalement** (statistiques de tout le train).

In [ ]:
# --- Cible : RUL plafonnee puis normalisee dans [0, 1] ---
for s in ["train", "val"]:
    TA[s]["rul_cap"] = np.minimum(TA[s]["rul"], RUL_MAX)
    TA[s]["cible"] = TA[s]["rul_cap"] / RUL_MAX

print("RUL brute train : min", int(TA["train"]["rul"].min()), "| max", int(TA["train"]["rul"].max()))
print("cible train dans [0,1] :", TA["train"]["cible"].min(), "-", TA["train"]["cible"].max())


In [ ]:
# --- Couches de normalisation (apprises sur l'entrainement uniquement) ---
# 1) une couche par regime, apprise sur les cycles sains du train (RUL plafonnee)
norm_regime = {}
for c in [0.6, 0.8, 1.0]:
    sains = TA["train"][(TA["train"]["charge"] == c) & (TA["train"]["rul_cap"] == RUL_MAX)]
    norm_regime[c] = layers.Normalization(axis=-1)
    norm_regime[c].adapt(sains[CAPTEURS].values.astype("float32"))

# 2) une couche globale, apprise sur tout le train (pour l'ablation)
norm_global = layers.Normalization(axis=-1)
norm_global.adapt(TA["train"][CAPTEURS].values.astype("float32"))

print("couches pretes : 3 par regime + 1 globale")


In [ ]:
# --- Application de la normalisation ---
def normaliser(donnees):
    """Normalisation par regime : un z-score par capteur et par charge."""
    d = donnees.copy()
    for c, couche in norm_regime.items():
        lignes = d["charge"] == c
        d.loc[lignes, CAPTEURS] = couche(d.loc[lignes, CAPTEURS].values.astype("float32")).numpy()
    return d

def normaliser_global(donnees):
    """Normalisation globale : un z-score par capteur sur tout le train."""
    d = donnees.copy()
    d[CAPTEURS] = norm_global(d[CAPTEURS].values.astype("float32")).numpy()
    return d

def xy(donnees, feats=FEATURES):
    """Retourne (entrees, cibles) d'un tableau deja normalise."""
    return donnees[feats].values.astype("float32"), donnees["cible"].values.astype("float32")

print("fonctions normaliser, normaliser_global et xy definies")


In [ ]:
# --- Entrees et cibles normalisees par regime ---
train_n = normaliser(TA["train"])
val_n = normaliser(TA["val"])

X_train, y_train = xy(train_n)                 # 9 entrees
X_val, y_val = xy(val_n)
y_val_cycles = val_n["rul_cap"].values.astype(float)     # RUL reelle en cycles

masque_coupes = val_n["rul_cap"].isin(DECOUPES).values   # points de coupe

print("X_train :", X_train.shape, "| X_val :", X_val.shape, "| points de coupe :", int(masque_coupes.sum()))


In [ ]:
# --- Memes donnees avec la normalisation globale (pour l'ablation) ---
train_g = normaliser_global(TA["train"])
val_g = normaliser_global(TA["val"])
print("versions normalisees pretes : regime (train_n/val_n) et globale (train_g/val_g)")


## 3. Metriques et suivi des experiences

In [ ]:
# --- Metriques d'evaluation ---
def rmse(y_vrai, y_pred):
    return float(np.sqrt(np.mean((y_vrai - y_pred) ** 2)))

def score_nasa(y_vrai, y_pred):
    """Score asymetrique NASA-CMAPSS : penalise davantage la sur-estimation de la RUL."""
    d = y_pred - y_vrai
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

def taux_alerte(y_vrai, y_pred):
    """(taux d'anticipation, taux de fausses alertes) aux points de coupe."""
    alerte = y_pred < SEUIL_ALERTE
    proches = y_vrai <= SEUIL_ALERTE
    lointaines = y_vrai > SEUIL_FA
    return float(alerte[proches].mean()), float(alerte[lointaines].mean())

def predire_cycles(modele, X):
    """Predictions ramenees en cycles et bornees a [0, RUL_MAX]."""
    return np.clip(RUL_MAX * modele.predict(X, verbose=0).ravel(), 0, RUL_MAX)

def evaluer(pred, y_cycles, masque=None):
    if masque is None:
        masque = np.ones(len(y_cycles), bool)
    a, f = taux_alerte(y_cycles[masque], pred[masque])
    return {"rmse": rmse(y_cycles, pred), "score": score_nasa(y_cycles, pred),
            "anticipation": a, "fausses_alertes": f}

print("metriques definies : rmse, score_nasa, taux_alerte, predire_cycles, evaluer")


In [ ]:
# --- Journal des experiences + suivi MLflow ---
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow_jalon1.db")
mlflow.set_experiment("uc5_jalon1_flotteA")
journal = []

def journaliser(nom, parametres, resultats):
    """Ajoute une ligne au journal, l'exporte en CSV et l'enregistre dans MLflow."""
    journal.append({"jalon": 1, "nom": nom, **parametres, **resultats})
    pd.DataFrame(journal).to_csv("journal_jalon1_flotteA.csv", index=False)
    numeriques = {k: float(v) for k, v in resultats.items() if np.isscalar(v) and np.isfinite(v)}
    with mlflow.start_run(run_name=nom):
        mlflow.log_params({k: str(v) for k, v in parametres.items()})
        mlflow.log_metrics(numeriques)
    print(nom, "->", {k: round(float(v), 2) for k, v in resultats.items()})
    return journal[-1]

print("journal pret")


In [ ]:
# --- Fabriques communes ---
def construire_modele(nb_entrees):
    """MLP 64-32-1 : architecture de reference."""
    return keras.Sequential([
        keras.Input(shape=(nb_entrees,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

def entrainer(modele, X_tr, y_tr, X_va, y_va, epochs=150, batch_size=256, perte="mse"):
    """Compile (Adam) et entraine le modele, sans progression."""
    modele.compile(optimizer="adam", loss=perte)
    modele.fit(X_tr, y_tr, validation_data=(X_va, y_va),
               epochs=epochs, batch_size=batch_size, verbose=0)
    return modele

print("fabriques definies : construire_modele, entrainer")


## 4. Modele de base (9 entrees)

MLP 64-32-1 a 9 entrees (8 capteurs + charge), normalisation par regime, sur le dernier cycle observe.

In [ ]:
# --- Modele de base : 9 entrees, normalisation par regime ---
base = construire_modele(X_train.shape[1])
entrainer(base, X_train, y_train, X_val, y_val)

pred_base = predire_cycles(base, X_val)
res_base = evaluer(pred_base, y_val_cycles, masque_coupes)
journaliser("base (9 entrees, regime)",
            {"normalisation": "regime", "perte": "mse", "features": 9}, res_base)


In [ ]:
# --- Erreur du modele de base en fonction de la RUL reelle ---
ecart = pred_base - y_val_cycles
plt.figure(figsize=(8, 4))
plt.scatter(y_val_cycles, ecart, s=4, alpha=0.3)
plt.axhline(0, color="red", linestyle=":")
plt.xlabel("RUL reelle (cycles, plafonnee a 125)")
plt.ylabel("RUL predite - RUL reelle")
plt.title("Erreur du modele de base")
plt.show()

print("biais moyen (predite - reelle) :", round(float(ecart.mean()), 2), "cycles")


**Lecture.** Le modele de base apprend correctement sur les cycles sains (RUL proche du plafond) mais se degrade pres de la panne, avec une tendance a **sur-estimer** la duree de vie : les predictions en retard, les plus couteuses au sens du score NASA.

## 5. Modele a 3 entrees vs modele de base (9 entrees)

L'EDA montre que **capteur_4** et **capteur_5** portent l'essentiel de l'information sur la RUL et que **charge** module les mesures. On entraine le **meme** MLP sur ces 3 entrees (normalisation par regime) et on le compare au modele de base a 9 entrees.

In [ ]:
# --- Modele a 3 entrees + normalisation par regime ---
X3_train, y3_train = xy(train_n, FEATURES_3)
X3_val, _ = xy(val_n, FEATURES_3)

modele_3 = construire_modele(3)
entrainer(modele_3, X3_train, y3_train, X3_val, y_val)

pred_3 = predire_cycles(modele_3, X3_val)
res_3 = evaluer(pred_3, y_val_cycles, masque_coupes)
journaliser("reduit (3 entrees, regime)",
            {"normalisation": "regime", "perte": "mse", "features": 3}, res_3)


In [ ]:
# --- Comparaison : modele de base (9 entrees) vs modele a 3 entrees ---
pd.DataFrame({
    "modele": ["base (9 entrees)", "reduit (3 entrees)"],
    "RMSE": [res_base["rmse"], res_3["rmse"]],
    "score": [res_base["score"], res_3["score"]],
    "anticipation": [res_base["anticipation"], res_3["anticipation"]],
    "fausses alertes": [res_base["fausses_alertes"], res_3["fausses_alertes"]],
})


In [ ]:
# --- Previsions sur la validation (dernier cycle par pompe) : 9 entrees vs 3 entrees ---
derniers = val_n.groupby("pompe").tail(1)
plt.figure(figsize=(8, 5))
plt.plot([0, RUL_MAX], [0, RUL_MAX], "k:", label="ideal")
plt.scatter(derniers["rul_cap"], pred_base[derniers.index.values], s=14, alpha=0.7, label="9 entrees")
plt.scatter(derniers["rul_cap"], pred_3[derniers.index.values], s=14, alpha=0.7, label="3 entrees")
plt.xlabel("RUL reelle (cycles)")
plt.ylabel("RUL predite (cycles)")
plt.title("Previsions sur la validation : 9 entrees vs 3 entrees")
plt.legend()
plt.show()


**Conclusion.** Le modele a **3 entrees** (`capteur_4`, `capteur_5`, `charge`) atteint une RMSE et un score au moins aussi bons que le modele de base a 9 entrees, pour un reseau beaucoup plus leger : **on retient le modele a 3 entrees.**

## 6. Normalisation : globale vs regime (3 entrees)

On garde le **meme** MLP a 3 entrees et on ne change que la normalisation des capteurs : **globale** (statistiques de tout le train) contre **par regime** (statistiques des cycles sains de chaque charge).

In [ ]:
# --- Modele a 3 entrees + normalisation GLOBALE ---
X3g_train, _ = xy(train_g, FEATURES_3)
X3g_val, _ = xy(val_g, FEATURES_3)

modele_3g = construire_modele(3)
entrainer(modele_3g, X3g_train, y3_train, X3g_val, y_val)

pred_3g = predire_cycles(modele_3g, X3g_val)
res_3g = evaluer(pred_3g, y_val_cycles, masque_coupes)
journaliser("reduit (3 entrees, globale)",
            {"normalisation": "globale", "perte": "mse", "features": 3}, res_3g)


In [ ]:
# --- Comparaison des normalisations (3 entrees) : globale vs regime ---
pd.DataFrame({
    "normalisation": ["globale", "regime"],
    "RMSE": [res_3g["rmse"], res_3["rmse"]],
    "score": [res_3g["score"], res_3["score"]],
    "anticipation": [res_3g["anticipation"], res_3["anticipation"]],
    "fausses alertes": [res_3g["fausses_alertes"], res_3["fausses_alertes"]],
})


In [ ]:
# --- Previsions sur la validation (dernier cycle par pompe) : globale vs regime ---
plt.figure(figsize=(8, 5))
plt.plot([0, RUL_MAX], [0, RUL_MAX], "k:", label="ideal")
plt.scatter(derniers["rul_cap"], pred_3g[derniers.index.values], s=14, alpha=0.7, label="3 entrees globale")
plt.scatter(derniers["rul_cap"], pred_3[derniers.index.values], s=14, alpha=0.7, label="3 entrees regime")
plt.xlabel("RUL reelle (cycles)")
plt.ylabel("RUL predite (cycles)")
plt.title("Previsions sur la validation : normalisation globale vs regime")
plt.legend()
plt.show()


**Conclusion.** La normalisation **par regime** est retenue : en retirant le niveau propre a chaque charge, elle fournit un signal plus physique et un score au moins equivalent a la normalisation globale. On conserve donc **3 entrees + normalisation par regime**, jeu sur lequel on compare les fonctions de perte ci-dessous.

## 7. Comparaison des fonctions de perte (3 entrees, normalisation par regime)

Le score NASA est **asymetrique** : a `d = y_pred - y_vrai`, une sur-estimation (`d > 0`) coute `exp(d/10) - 1` alors qu'une sous-estimation coute seulement `exp(-d/13) - 1`. On compare, sur le meme MLP a 3 entrees normalise par regime :

- **MSE** : reference symetrique (modele `modele_3` ci-dessus) ;
- **asymetrique ponderee** : sur-estimation penalisee `PENALITE` fois ;
- **inspiree du score** : perte exponentielle calquee sur le score NASA.

In [ ]:
# --- Fonctions de perte asymetriques ---
PENALITE_SURESTIMATION = 3.0

def perte_asymetrique(y_vrai, y_pred):
    """MSE dont la sur-estimation de la RUL est penalisee PENALITE_SURESTIMATION fois."""
    erreur = y_pred - y_vrai
    return tf.reduce_mean(tf.where(erreur > 0,
                                   PENALITE_SURESTIMATION * tf.square(erreur),
                                   tf.square(erreur)))

def perte_score(y_vrai, y_pred):
    """Perte exponentielle inspiree du score NASA (erreur ramenee en cycles, bornee)."""
    d = tf.clip_by_value((y_pred - y_vrai) * RUL_MAX, -30.0, 30.0)
    return tf.reduce_mean(tf.where(d < 0, tf.exp(-d / 13.0) - 1.0, tf.exp(d / 10.0) - 1.0))

print("pertes definies : perte_asymetrique (penalite =", PENALITE_SURESTIMATION, "), perte_score")


In [ ]:
# --- Comparaison MSE vs pertes asymetriques (3 entrees, regime) ---
modele_asym = construire_modele(3)
entrainer(modele_asym, X3_train, y3_train, X3_val, y_val, perte=perte_asymetrique)
pred_asym = predire_cycles(modele_asym, X3_val)
res_asym = evaluer(pred_asym, y_val_cycles, masque_coupes)
journaliser("reduit (3 entrees) - perte asymetrique",
            {"normalisation": "regime", "perte": "asymetrique", "features": 3}, res_asym)

modele_score = construire_modele(3)
entrainer(modele_score, X3_train, y3_train, X3_val, y_val, perte=perte_score)
pred_score = predire_cycles(modele_score, X3_val)
res_score = evaluer(pred_score, y_val_cycles, masque_coupes)
journaliser("reduit (3 entrees) - perte inspiree du score",
            {"normalisation": "regime", "perte": "score", "features": 3}, res_score)

pd.DataFrame({
    "perte": ["mse", "asymetrique", "inspiree du score"],
    "RMSE": [res_3["rmse"], res_asym["rmse"], res_score["rmse"]],
    "score": [res_3["score"], res_asym["score"], res_score["score"]],
    "anticipation": [res_3["anticipation"], res_asym["anticipation"], res_score["anticipation"]],
    "fausses alertes": [res_3["fausses_alertes"], res_asym["fausses_alertes"], res_score["fausses_alertes"]],
})


In [ ]:
# --- Diagnostic : d'ou vient le score ? (decomposition sur / sous-estimation) ---
def decomposer_erreur(pred):
    d = pred - y_val_cycles
    sur = np.where(d > 0, np.exp(d / 10) - 1, 0.0)
    sous = np.where(d < 0, np.exp(-d / 13) - 1, 0.0)
    return pd.Series({"sur-estimation": sur.sum(), "sous-estimation": sous.sum(),
                      "total": sur.sum() + sous.sum(), "nb |erreur| > 20": int((np.abs(d) > 20).sum())})

pd.DataFrame({"mse": decomposer_erreur(pred_3),
              "asymetrique": decomposer_erreur(pred_asym),
              "inspiree du score": decomposer_erreur(pred_score)}).T


**Conclusion.** La perte asymetrique reduit la part de **sur-estimation** du score (predictions en retard, les plus couteuses), au prix parfois d'une RMSE legerement moins bonne. La perte exponentielle inspiree du score, en penalisant tres fort quelques outliers, est en general instable. On retient la perte asymetrique : **la configuration utilisee dans la suite du notebook est 3 entrees + normalisation par regime + perte asymetrique.**